# Unit III: Text as Data

## Is climate change an endangered-species threat? Reading thirty years of the Endangered Species Act

In 2019, [Delach et al.](https://doi.org/10.1038/s41558-019-0620-8) asked whether the agencies
that implement the U.S. Endangered Species Act take climate change seriously. They read the
official documents for all 459 endangered animals by hand. Climate change was named as a threat
for 64% of them, and management actions to address it were planned for 18%. Their per-species
codings are [openly published](https://doi.org/10.17605/OSF.IO/R9UCA).

That took a team of people many months. In this module you will read a related body of
documents with code, then check your machine reading against theirs.

Previous modules treated tables and maps as data. This one treats **text** as data. You will
climb a ladder of tools, each answering a sharper version of the same question:

| Part | Tool | Question it can answer |
|---|---|---|
| 1 | regular expressions | Where is climate change *mentioned*? |
| 2 | a classifier ([Laya](https://huggingface.co/convaiinnovations/laya)) | Which mentions are *findings* that it threatens the species? |
| 3 | embeddings | Where is a threat *described*, whatever words are used? |
| 4 | retrieval-augmented generation | Can I ask the documents a question and check the answer? |
| 5 | structured extraction | One row per decision: what did the agency conclude, and why? |

Every tool past Part 1 is a model, and every model is sometimes wrong. The question from Unit I
still applies, with more force:

> **How do I know these numbers are right?**

## The legal record

Under section 4 of the ESA, a species is listed as endangered or threatened because of one or
more of five factors:

- **(A)** the present or threatened destruction, modification, or curtailment of its habitat or range
- **(B)** overutilization for commercial, recreational, scientific, or educational purposes
- **(C)** disease or predation
- **(D)** the inadequacy of existing regulatory mechanisms
- **(E)** other natural or manmade factors affecting its continued existence

The decision must be made "solely on the basis of the best scientific and commercial data
available." Each final listing decision is published as a rule in the
[Federal Register](https://www.federalregister.gov/), the daily journal of the U.S. government.
The rule lays out the evidence, responds to public and peer-review comments, and states the
agency's conclusion. Those rules are legal documents, so they can be challenged in court, and
they are written with that in mind.

That makes them an unusually good corpus. Every rule is public, they follow a common structure,
and they span decades. They are also long, repetitive, and full of hedged legal prose, which is
exactly why nobody reads all of them.

## How this notebook works

Every part has **baseline tasks** that everyone completes. Tasks marked **🧗 Go deeper** are
optional. They are where strong analyses distinguish themselves, and some of them are genuinely
open research questions. Do the baseline for every part before going deeper in any one.

### Rules of engagement

- **Plan mode only.** You review and approve the plan before any code runs.
- **Keep your prompts.** Record what you asked for. The prompt is part of your method.
- **Never report a number you have not checked.** This includes numbers produced by a classifier
  or a language model. Model output is data to be verified, not an answer.
- **Every claim traces to a paragraph.** When a model tells you what a rule says, you must be
  able to point to the paragraph that says it.

Setup, GitHub authentication, and model configuration are on the
[course website](https://espm-157.carlboettiger.info/).

### Verification block

Parts 1, 2 and 5 end with a verification block. Answer each item in your own words:

1. **Extent:** how many documents or paragraphs, and from what years?
2. **Coverage:** what is missing from what you analyzed, and does it matter?
3. **Agreement:** where did two methods (or you and a model) disagree, and who was right?
4. **Error:** one concrete case where a tool was wrong, and how you found it.
5. **Cross-check:** one result confirmed against an independent source.

In [ ]:
import os, re, pathlib
import requests
import pandas as pd
import duckdb
from plotnine import *

### A language model, as a function

You have used a language model as a coding assistant. The same model is also an API: a function
you can call from code. Here we use `deepseek-v4-flash`, served by the
[National Research Platform](https://nrp.ai/documentation/userdocs/ai/llm-managed/), through
[LangChain](https://python.langchain.com/). We turn the model's "thinking" off for speed, and set
`temperature=0` to make its answers as repeatable as possible.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

nrp = dict(base_url="https://ellm.nrp-nautilus.io/v1", api_key=os.environ["NRP_TOKEN"])
llm = ChatOpenAI(model="deepseek-v4-flash", temperature=0, timeout=180,
                 extra_body={"thinking": {"type": "disabled"}}, **nrp)
embedder = OpenAIEmbeddings(model="qwen3-embedding", check_embedding_ctx_length=False, **nrp)

In [ ]:
llm.invoke("In one sentence, what does the U.S. Endangered Species Act do?").content

---

# Part 1: The corpus, and what regular expressions can tell you

The Federal Register has an open API, which needs no key. This query asks for every Fish and
Wildlife Service rule containing the phrase that titles ESA actions, one year at a time.

In [ ]:
def federal_register_rules(year):
    params = {"conditions[agencies][]": "fish-and-wildlife-service",
              "conditions[type][]": "RULE",
              "conditions[term]": '"Endangered and Threatened Wildlife and Plants"',
              "conditions[publication_date][year]": year,
              "per_page": 1000,
              "fields[]": ["document_number", "publication_date", "title", "raw_text_url"]}
    return requests.get("https://www.federalregister.gov/api/v1/documents.json", params=params).json().get("results", [])

rules = pd.DataFrame([r for year in range(1994, 2027) for r in federal_register_rules(year)])
rules["year"] = rules.publication_date.str[:4].astype(int)
rules

Not all of these are listing decisions. Many designate critical habitat, remove species from
the list, or correct earlier rules. A title filter written with a model's help:

In [ ]:
listing_pattern = r"(?:Endangered|Threatened) (?:Species )?Status|Determination of (?:Endangered|Threatened)|Listing"
exclude_pattern = r"Removal|Delist|Reclassif|Critical Habitat|Proposed|Withdrawal"

listings = rules[rules.title.str.contains(listing_pattern) & ~rules.title.str.contains(exclude_pattern)].copy()
len(listings)

### Task 1.1: Audit the regular expression

A regular expression is a precise, literal pattern. That makes it easy to check and easy to get
wrong. Read titles that the filter **drops** as well as ones it keeps. Find at least two
kinds of title it misclassifies, count them, and fix the patterns. Report the corrected count.

A regex that drops the wrong rules does not fail. It runs, and the plot just quietly changes.

**Your answer:**

### Getting the text

Each rule's full text is published as plain text. This downloads them once into `data/`.

In [ ]:
text_dir = pathlib.Path("data/rules")
text_dir.mkdir(parents=True, exist_ok=True)

def rule_text(row):
    path = text_dir / f"{row.document_number}.txt"
    if not path.exists():
        path.write_text(requests.get(row.raw_text_url).text)
    return path.read_text()

listings["raw"] = [rule_text(row) for row in listings.itertuples()]
print(listings.raw.iloc[0][:3000])

### From documents to paragraphs

Text has to be cut into units before you can count, classify, or search it. Look at the raw
text above: HTML tags, page markers like `[[Page 20948]]`, lines wrapped at a fixed width, and
paragraphs that start with an indent. This code, written with a model's help, turns every rule
into paragraphs: one row per paragraph, with the rule's `doc` number, its `year`, the
paragraph's position `para`, and its `text`.

In [ ]:
def split_paragraphs(raw):
    body = re.sub(r"<[^>]+>", "", raw)
    body = re.sub(r"\[\[Page \d+\]\]", "", body)
    chunks = re.split(r"\n\s*\n|\n(?= {4}\S)", body)
    return [" ".join(chunk.split()) for chunk in chunks if len(chunk.split()) > 20]

paras = pd.DataFrame([{"doc": rule.document_number, "year": rule.year, "para": i, "text": text}
                      for rule in listings.itertuples()
                      for i, text in enumerate(split_paragraphs(rule.raw))])
paras

### Task 1.2: Audit the paragraphs

Every later step treats a row of `paras` as one unit of meaning, so check that it is one.

Open one rule on federalregister.gov and compare a stretch of the rendered page with your rows.
Then answer with evidence from the data, not by eye alone:

- What does `len(chunk.split()) > 20` throw away? Count it, and look at what it is.
- Section headings in these rules are short lines. Where do they end up? Find an example.
- Would either problem change a count of "paragraphs that mention climate change"?

**Your answer:**

### Task 1.3: When did climate change enter the record?

For each rule, count the paragraphs that mention climate change. Plot the share of listing
rules that mention it at all, by year. Choose your pattern deliberately: does it catch
`Climate Change` at the start of a heading? `climatic change`? Should it?

Describe the trend. Name the year it changes, and look at the most recent rules too.

**Your answer:**

### 🧗 Task 1.4: Go deeper: mechanisms

Climate change threatens species through specific mechanisms: sea-level rise, drought, warmer
water, lost snowpack, more intense fire. Write patterns for three of them and track them over
time. Which spellings and hyphenations did your first attempt miss? How would you know what
you missed?

### 🔍 Verification block 1

1. Extent:
2. Coverage:
3. Agreement:
4. Error:
5. Cross-check:

---

# Part 2: A mention is not a finding

Delach et al. coded whether climate change was **identified as a threat** to a species. That is
not the same as appearing in the document. A rule can mention climate change in a legal
background section, in a response to a public comment, or to conclude that it is *not* a
significant threat.

Telling those apart takes judgement. Doing it for thousands of paragraphs takes a classifier.

### Task 2.1: Label some paragraphs yourself

Before you trust any model, you need a small set of answers you trust. Label each paragraph in
this sample as one of `threat finding`, `background`, `dismissal`, or `comment response`.
Read the whole paragraph each time. Note where you hesitated: hesitation marks paragraphs that
are genuinely ambiguous, and a model cannot do better than the ambiguity in the text.

In [ ]:
sample = paras[paras.text.str.contains(r"(?i)climat(?:e|ic) change")].sample(20, random_state=157)
for row in sample.itertuples():
    print(f"[{row.Index}] {row.year}  {row.text}\n")

**Your answer:**

### A decision model

[Laya](https://huggingface.co/convaiinnovations/laya) is an open-weights *decision model*. It is
an open alternative to commercial decision APIs like TypeSafe's Jev. You give it a text (the
`state`) and typed questions, and it returns a **probability** for each possible answer in one
pass. It never generates text, so there is nothing to parse and nothing to hallucinate. It can
still be wrong.

- A `noul` question returns a probability that the answer is yes.
- A `choice` question returns a probability for each label you define.

Laya runs on a GPU on the National Research Platform, and you call it over HTTP, the same way
you call the language model. (It is too large to run comfortably inside your 4 GB session.) You need your own `LAYA_API_KEY`,
set in your environment like `NRP_TOKEN`.

In [ ]:
laya_url = "https://laya.nrp-nautilus.io"
laya_auth = {"Authorization": f"Bearer {os.environ['LAYA_API_KEY']}"}

def decide(text, questions):
    response = requests.post(f"{laya_url}/v1/systemone", headers=laya_auth,
                             json={"state": text, "questions": questions})
    response.raise_for_status()
    return response.json()["answers"]

questions = {
    "threat": {"type": "noul",
               "instructions": "Does this paragraph conclude that climate change is a threat to the species being listed?"},
    "role": {"type": "choice",
             "instructions": "What is this paragraph doing?",
             "criteria": {"threat finding": "concludes climate change harms or will harm the species or its habitat",
                          "background": "general or legal background about climate change, not about this species",
                          "dismissal": "considers climate change and concludes it is not a significant threat",
                          "comment response": "responds to a public or peer reviewer comment"}}}

decide(sample.text.iloc[0], questions)

In [ ]:
answers = [decide(text, questions) for text in sample.text]
sample["p_threat"] = [a["threat"]["noul"] for a in answers]
sample["laya_role"] = [a["role"]["choice"] for a in answers]
sample[["year", "my_label", "laya_role", "p_threat"]]

### Task 2.2: Who is right?

Cross-tabulate your labels against Laya's. Read every disagreement again. For each, decide who
was right, you or the model, and say why.

Then check the probabilities. A model is *calibrated* if, among paragraphs it gives
`p_threat` ≈ 0.8, about 80% really are threat findings. With twenty paragraphs you cannot
measure that precisely. What can you say?

**Your answer:**

### Task 2.3: A first pass over the corpus

Classifiers are most useful as a **filter**: a cheap first pass that decides which paragraphs
deserve a closer look. Run the `threat` question over every climate-mentioning paragraph.
(Each call takes a fraction of a second, so this takes a few minutes. Save the results to
`data/` so you only do it once.) Choose a threshold on `p_threat` and defend it using your hand
labels.

Then remake the Task 1.3 figure, counting a rule only if at least one paragraph passes your
threshold. Plot both lines together. How big is the gap between *mentioned* and *found to be a
threat*, and has it changed over time?

**Your answer:**

### 🧗 Task 2.4: Go deeper: a language model as a classifier

Ask `llm` the same yes/no question about your 20 hand-labelled paragraphs. You can recover a
probability from a language model too: request `logprobs` and look at the probability it gave
to the token `Yes`. Compare it with Laya on accuracy against your labels, on time per paragraph,
and on what each would cost to run over all 40,000+ paragraphs.

### 🔍 Verification block 2

1. Extent:
2. Coverage:
3. Agreement:
4. Error:
5. Cross-check:

---

# Part 3: Searching by meaning

A regular expression finds words. An **embedding** turns a paragraph into a long list of
numbers (here, 4096 of them) placed so that paragraphs with similar *meaning* end up close
together. "Rising seas will inundate the nesting beaches" and "coastal habitat loss from
sea-level rise" share almost no words, but their embeddings are close.

We embed every paragraph of the listing rules published since 2020. That takes several minutes,
so the vectors are saved to `data/embeddings/` in batches and never computed twice. DuckDB then
searches them straight from disk: a search is just a query sorted by cosine similarity.

In [ ]:
import numpy as np

index_dir = pathlib.Path("data/embeddings")
index_dir.mkdir(exist_ok=True)
recent = paras[paras.year >= 2020][["doc", "year", "para", "text"]]
for start in range(0, len(recent), 500):
    path = index_dir / f"{start:06d}.parquet"
    if not path.exists():
        batch = recent.iloc[start:start + 500]
        vectors = np.array(embedder.embed_documents(batch.text.tolist()), dtype="float32")
        batch.assign(embedding=list(vectors)).to_parquet(path)

con = duckdb.connect()
con.sql(f"create or replace view vectors as select doc, year, para, text, embedding::FLOAT[4096] as embedding from '{index_dir}/*.parquet'")
con.sql("select count(*) as paragraphs, count(distinct doc) as rules from vectors")

In [ ]:
def search(query, k=8):
    return con.sql('''
        select doc, year, para, text, array_cosine_similarity(embedding, $q::FLOAT[4096]) as similarity
        from vectors order by similarity desc limit $k
    ''', params={"q": embedder.embed_query(query), "k": k}).df()

search("rising seas will flood coastal habitat")

### Task 3.1: Meaning versus words

Choose one climate mechanism. Search for it two ways: with a regular expression over the same
paragraphs (`recent`), and with `search()`. List paragraphs that one method finds and the other
misses. What does each method get wrong? Which would you use to *count* things, and which to
*find* things?

**Your answer:**

### 🧗 Task 3.2: Go deeper: copy and paste

Some paragraphs appear, nearly word for word, in rule after rule. Use the embeddings to find
pairs of paragraphs from *different* rules with cosine similarity above 0.97 (DuckDB can
compute this as a self-join). What fraction of climate-change text is shared boilerplate?
What does that do to a measure of "agency attention" built on counting mentions, as in Part 1?

---

# Part 4: Asking the documents

**Retrieval-augmented generation (RAG)** combines Parts 3 and the language model. It *retrieves*
the paragraphs most relevant to a question, then asks the model to answer *using only those
paragraphs*, citing them by id. The model supplies fluency; the retrieved text supplies the
facts. At least, that is the idea.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about U.S. Endangered Species Act listing rules using only the "
               "paragraphs provided. Cite the id of the supporting paragraph in square brackets after "
               "every claim, like [2024-01234:57]. If the paragraphs do not answer the question, say so."),
    ("human", "Paragraphs:\n\n{context}\n\nQuestion: {question}"),
])
rag_chain = rag_prompt | llm

def ask(question, k=8):
    hits = search(question, k)
    context = "\n\n".join(f"[{row.doc}:{row.para}] {row.text}" for row in hits.itertuples())
    return rag_chain.invoke({"context": context, "question": question}).content

answer = ask("How is drought expected to affect freshwater species listed since 2020?")
print(answer)

### Task 4.1: Check every citation

Ask two questions of your own about how climate change threatens listed species. For every
claim in each answer, look up the cited paragraph (`paras` has them all) and record whether it
**supports**, **partly supports**, or **does not support** the claim.

| question | claim | cited | verdict |
|---|---|---|---|
| | | | |

What fraction of claims were fully supported? Did the model ever cite the right paragraph but
say more than it does?

**Your answer:**

### Task 4.2: Ask about something that is not there

The index holds only rules published since 2020. The polar bear was listed in 2008, the first
species listed primarily because of climate change. Ask the system why the polar bear was listed.

The model has almost certainly read about the polar bear in its training data. Does it answer
from the paragraphs it was given, or from memory? Is either answer acceptable here? How would a
reader of the answer know the difference?

**Your answer:**

### 🧗 Task 4.3: Go deeper: what retrieval controls

Change `k`, or rephrase one question three ways. Does the answer change? Find a question where
retrieval, not the language model, is clearly the weak link, and explain how you can tell.

---

# Part 5: One row per decision

Part 2 classified paragraphs. To compare with Delach et al. we need a judgement about each
**rule**: which species it lists, and what it concludes about climate change. This is
**structured extraction**. We define a schema with [Pydantic](https://docs.pydantic.dev/), and
the model must fill in exactly those fields, with the allowed values, every time.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ClimateAssessment(BaseModel):
    species: list[str] = Field(description="scientific names of every species listed by this rule")
    taxon_group: Literal["mammal", "bird", "reptile", "amphibian", "fish", "insect", "mollusk",
                         "crustacean", "arachnid", "plant", "multiple", "other"]
    climate_role: Literal["threat to the species", "considered, not a significant threat", "not discussed"]
    mechanisms: list[Literal["sea level rise", "drought", "higher temperatures", "altered precipitation",
                             "storms", "wildfire", "snowpack loss", "ocean acidification", "other"]]
    evidence: str = Field(description="one sentence copied verbatim from the text that supports climate_role")

extract = llm.with_structured_output(ClimateAssessment, method="function_calling")

A whole rule can run to 100,000 words, more than we want to send for every decision. Instead
we send the title, the opening paragraphs (where species are introduced), and the paragraphs
that mention climate change. That choice is itself a filter, and Task 5.1 asks what it costs.

In [ ]:
def rule_context(doc):
    rule = paras[paras.doc == doc]
    title = listings.set_index("document_number").title[doc]
    excerpt = pd.concat([rule.head(4), rule[rule.text.str.contains(r"(?i)climate change")].head(20)]).drop_duplicates("para")
    return f"Final ESA listing rule: {title}\n\n" + "\n\n".join(excerpt.text)

doc = listings.document_number.iloc[150]
extract.invoke(rule_context(doc))

### Task 5.1: Extract, then audit

Run `extract` on 10 rules spread across the years, including some from before 2005, and collect
the results into a DataFrame. Then audit every row against the rule itself:

- Is the `evidence` sentence actually in the text? (Check with code, not by eye.)
- Is the species list complete? Compare with the rule's title and summary.
- Do you agree with `climate_role`? Use what you learned labelling paragraphs in Part 2.

What kinds of error did you find, and which fields would you trust at scale?

**Your answer:**

### Task 5.2: Three measurements of one question

Combine your results into one figure that compares, over time, the share of rules that
**mention** climate change (Part 1), that the **classifier** says find it a threat (Part 2),
and that the **extraction** says find it a threat (Part 5). If you extracted only a sample of
rules, show that honestly.

Which line would you report as "the share of ESA listings that identify climate change as a
threat," and why?

**Your answer:**

### Task 5.3: Reconcile with Delach et al.

Their per-species data: `https://osf.io/download/76ytr/` (the column meanings are in
`https://osf.io/download/4bm8u/`).

Their headline number is 64%. Yours will differ. Before you compare, establish what each
number counts:

- Which **documents** did they read for each species? Which did you?
- Which **species**: endangered animals only, or everything? FWS only, or NMFS too?
- Which **years**?

Then match species between the two datasets and compare `CC_Discuss` with your extraction for
the species you have in common. Be careful: a species *named* in a rule is not necessarily the
species the rule *lists*.

**For the matched species, how often do you agree? Where you disagree, is it an error, or are
you measuring different things?**

**Your reconciliation:**

### 🧗 Task 5.4: Go deeper

Choose one:

- **Scale.** Extract every listing rule, caching results to `data/` as you go. Re-run 10 of them
  and report how many fields change between runs.
- **Schema.** Add a field for the listing factor (A–E) under which climate change is analyzed.
  Does the agency treat climate change mainly as habitat loss (A) or as an "other" factor (E)?
  Has that changed?
- **Method.** We used `method="function_calling"`. Try `"json_schema"` and `"json_mode"` on the
  same rules and compare speed, validity, and accuracy.
- **The recent record.** Describe what the rules from 2025 onward say about climate change,
  and how it differs from rules a decade earlier. Support every claim with a paragraph id.

### 🔍 Verification block 3

1. Extent:
2. Coverage:
3. Agreement:
4. Error:
5. Cross-check:

---

# Part 6: Reflection

1. **The ladder.** For each tool (regex, classifier, embeddings, RAG, extraction), name one
   thing it did well here and one way it failed.
2. **The 64%.** In two or three sentences for a non-scientist, how often does the U.S.
   government identify climate change as a threat when it lists a species? How confident are
   you, and what would change your answer?
3. **Model as tool, model as author.** The same model wrote your code and was called *by* your
   code. Where did you have to check it in each role? Which was harder to check?
4. **Division of labor.** What did you contribute that the models did not? If the answer for
   some task is "nothing", say so.
5. **Legal text.** Listing rules are written to survive court challenge. How did that shape the
   language, and what did it do to your tools?

**Your reflection:**